# 2.4 Cube 计算指令：矩阵乘的指挥棒

货物（矩阵）就位到 L0A/L0B 后，轮到 **mmad** 指令登场。本节是第四块积木——FA 两大主力计算（QK^T、P·V）的执行细节。

## 1. 脉动阵列回顾

1.2 节已经看过 Cube 的物理结构（[点此回看](01.02_ascend_950_arch.ipynb)），这里只复习三个要点：

1. **16×16×16**：一个 Cube 单位周期完成 4096 次 MAC；
2. **数据流式通过**：矩阵元素沿三个方向流动，像流水线上的工件；
3. **输入必须分形**：L0A/L0B 中的数据须为 NZ 等分形格式（MTE1 搬运时已转换）；
4. **算力的微观来源**：一拍 4096 次 MAC × 核数 × 主频，凑出 1.2 节 354T/433T/486T 的算力阶梯；作为对照，Vector 一拍做 128 个 FP16 加法——一个数量级的差距，正是 8:1 算力配比的微观解释。

<img src="./images/cube_systolic_array.png" width="560">

## 2. mmad 指令与 M/N/K

矩阵乘 C = A×B 的维度语言：A 是 **M×K**，B 是 **K×N**，C 是 **M×N**。mmad 沿 K 维做累加：

```text
for k_block in 0..K/16:            # K 维逐块推进
    mmad(C_fractal,                # 累加目标：L0C 中的分形
         A_fractal[k_block],       # L0A 中的 16×16 块
         B_fractal[k_block],        # L0B 中的 16×16 块
         init_flag)                # 首块清零 / 后续累加
```

> **三个实践要点**：
> 1. **init_flag**：K 维第一块清零、后续块累加——FA 的 P·V 累加直接复用此语义；
> 2. **K 维切分**（一次放不下 K 时把 K 拆段循环）是 Cube 调优的核心旋钮（第8章“K 切分”）；
> 3. 高阶 MatMul API（`matmul` 等）会自动完成分形、循环与尾块处理——第3篇用高阶 API 写 FA，但**看懂 Profiler 的前提是理解 mmad**。

## 3. Cube 支持的数据类型

950 的 Cube 大幅扩展了精度格式：

<img src="./images/cube_data_types.png" width="560">

> **看图要点**：从经典 FP32/FP16/BF16/INT8，到 950 新增的 **HiF8 / MXFP8 / FP8 / MXFP4**。低精度 = 更高算力（MXFP4 峰值为 FP16 的 4 倍）+ 更低带宽占用。FA 的 QK^T 与 P·V 是全模型中算力最重的部分之一，正是这些新格式的首要受益者——第6章全量化 FA 将直接使用。

## 4. Cube 编程范式：五段式

Vector 算子是三段式（CopyIn/Compute/CopyOut），Cube 算子多两步——**Split/Aggregate**：

<img src="./images/cube_programming_paradigm.png" width="600">

> **看图要点**：**CopyIn**（GM→L1）→ **Split**（L1→L0A/B，分形）→ **Compute**（mmad 循环）→ **Aggregate**（L0C→UB，聚合成常规排布）→ **CopyOut**（UB→GM）。多出的两步正是 Cube 的“格式税”：进出矩阵乘都要转换排布。FA Kernel 里两次矩阵乘都要交这笔税——950 的 CV 融合与 FIXPIPE 随路转换正是为了削减它。

**流水线视角**：一次 Cube 计算要穿越 MTE2 → MTE1 → mmad → FIXPIPE **四条流水线**——段数多、每段都可能成瓶颈，第7~8章的 Cube 类调优大多在打磨这四段的重叠。

完整数据流图（与 2.1 节呼应，建议对照）：

<img src="./images/matmul_compute_flow.png" width="600">

## 小结

| 关键词 | 一句话 |
|--|--|
| mmad | 一条指令完成 16×16×16 乘加，K 维逐块累加 |
| M/N/K | 描述矩阵乘的三维语言，K 切分是调优旋钮 |
| 数据类型 | 950 新增 HiF8/MXFP8/FP8/MXFP4，低精度即高算力 |
| 五段式 | CopyIn → Split → Compute → Aggregate → CopyOut |

下一节 [2.5 Vector 计算指令](01.07_vector_instructions.ipynb)：夹在两次矩阵乘之间的 Softmax 交给谁。

## 课后练习

1.（单选）mmad 指令的累加目标存放于？（A. L0A　B. L0B　C. L0C　D. UB）

2.（判断）使用高阶 MatMul API 时，程序员仍需手动编写 ND → NZ 的分形转换代码。

3.（单选）FA 中 P·V 的多块累加，可以直接复用 mmad 的哪个机制？（A. 尾块 mask　B. init_flag 首块清零后续累加　C. 双缓冲　D. 随路量化）

> 完成后查看 [answer/01.06_answer.txt](answer/01.06_answer.txt) 核对答案。